# **Predicting Used Car Selling Price Using Machine Learning**

Team members:

1.   Heng-Jui Chu
2.   Simbarashe Mpofu
3.   Simbarashe Simbangegavi

In [1]:
from google.colab import drive
import sys

# Mount Google Drive
drive.mount('/content/drive')

# Get the absolute path of the current folder
abspath_curr = '/content/drive/My Drive/Colab Notebooks/ML project/'

# # Get the absolute path of the shallow utilities folder
# abspath_util_shallow = '/content/drive/My Drive/Colab Notebooks/teaching/gwu/machine_learning_I/code/utilities/p2_shallow_learning/'

# # Get the absolute path of the shallow models folder
# abspath_model_shallow = '/content/drive/My Drive/Colab Notebooks/teaching/gwu/machine_learning_I/code/models/p2_shallow_learning/'

Mounted at /content/drive


**Warning**

In [2]:
import warnings

# Ignore warnings
warnings.filterwarnings('ignore')

**Matplotlib**

In [3]:
import matplotlib.pyplot as plt
%matplotlib inline

# Set matplotlib sizes
plt.rc('font', size=20)
plt.rc('axes', titlesize=20)
plt.rc('axes', labelsize=20)
plt.rc('xtick', labelsize=20)
plt.rc('ytick', labelsize=20)
plt.rc('legend', fontsize=20)
plt.rc('figure', titlesize=20)

**TensorFlow**

In [4]:
# The magic below allows us to use tensorflow version 2.x
%tensorflow_version 2.x
import tensorflow as tf
from tensorflow import keras

Colab only includes TensorFlow 2.x; %tensorflow_version has no effect.


**Random seed**

In [5]:
# The random seed
random_seed = 42

# Set random seed in tensorflow
tf.random.set_seed(random_seed)

# Set random seed in numpy
import numpy as np
np.random.seed(random_seed)

## **Import tools**

In [7]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from xgboost import XGBRegressor
import shap


**Data Preprocessing**

**Loading the data**

In [11]:
df_raw_train = pd.read_csv(abspath_curr + "used_car_dataset.csv")

# Make a copy of df_raw_train
df_train = df_raw_train.copy(deep=True)

# Get the name of the target
target = 'price_usd'

In [12]:
# Print the dimension of df_train
pd.DataFrame([[df_train.shape[0], df_train.shape[1]]], columns=['# rows', '# columns'])

,# rows,# columns
0,10000,12


In [13]:
# Print the first 5 rows of df_train
df_train.head()

,make_year,mileage_kmpl,engine_cc,fuel_type,owner_count,price_usd,brand,transmission,color,service_history,accidents_reported,insurance_valid
0,2001,8.17,4000,Petrol,4,8587.64,Chevrolet,Manual,White,NaN,0,No
1,2014,17.59,1500,Petrol,4,5943.50,Honda,Manual,Black,NaN,0,Yes
2,2023,18.09,2500,Diesel,5,9273.58,BMW,Automatic,Black,Full,1,Yes
3,2009,11.28,800,Petrol,1,6836.24,Hyundai,Manual,Blue,Full,0,Yes
4,2005,12.23,1000,Petrol,2,4625.79,Nissan,Automatic,Red,Full,0,Yes


**Splitting the data**

The code below shows how to divide the training data into training (60%) ,validation (20%) and testing (20%).

In [14]:
from sklearn.model_selection import train_test_split

# Divide the training data into training (60%) , temp (40%)
df_train, df_temp = train_test_split(df_train, train_size=0.6, random_state=random_seed)

# Second split: split temp into validation and test (20% each)
df_val, df_test = train_test_split(df_temp, test_size=0.5, random_state=random_seed)

# Reset the index
df_train, df_val , df_test = df_train.reset_index(drop=True), df_val.reset_index(drop=True) , df_test.reset_index(drop=True)

In [15]:
# Print the dimension of df_train
pd.DataFrame([[df_train.shape[0], df_train.shape[1]]], columns=['# rows', '# columns'])

,# rows,# columns
0,6000,12


In [16]:
# Print the dimension of df_val
pd.DataFrame([[df_val.shape[0], df_val.shape[1]]], columns=['# rows', '# columns'])

,# rows,# columns
0,2000,12


In [17]:
# Print the dimension of df_test
pd.DataFrame([[df_test.shape[0], df_test.shape[1]]], columns=['# rows', '# columns'])

,# rows,# columns
0,2000,12


**Identifying uncommon variables**

The code below shows how to find common variables between the training, validation and test data.

In [18]:
def common_var_checker(df_train, df_val, df_test, target):
    """
    The common variables checker

    Parameters
    ----------
    df_train : the dataframe of training data
    df_val : the dataframe of validation data
    df_test : the dataframe of test data
    target : the name of the target

    Returns
    ----------
    The dataframe of common variables between the training, validation and test data
    """

    # Get the dataframe of common variables between the training, validation and test data
    df_common_var = pd.DataFrame(np.intersect1d(np.intersect1d(df_train.columns, df_val.columns), np.union1d(df_test.columns, [target])),
                                 columns=['common var'])

    return df_common_var

In [19]:
# Call common_var_checker
#
df_common_var = common_var_checker(df_train, df_val, df_test, target)

# Print df_common_var
df_common_var

,common var
0,accidents_reported
1,brand
2,color
3,engine_cc
4,fuel_type
5,insurance_valid
6,make_year
7,mileage_kmpl
8,owner_count
9,price_usd


The code below shows how to find features in the training data but not in the validation or test data.

In [20]:
# Get the features in the training data but not in the validation or test data
uncommon_feature_train_not_val_test = np.setdiff1d(df_train.columns, df_common_var['common var'])

# Print the uncommon features
pd.DataFrame(uncommon_feature_train_not_val_test, columns=['uncommon feature'])

,uncommon feature


The code below shows how to find the features in the validation data but not in the training or test data.

In [21]:
# Get the features in the validation data but not in the training or test data
uncommon_feature_val_not_train_test = np.setdiff1d(df_val.columns, df_common_var['common var'])

# Print the uncommon features
pd.DataFrame(uncommon_feature_val_not_train_test, columns=['uncommon feature'])

,uncommon feature


The code below shows how to find the features in the test data but not in the training or validation data.

In [22]:
# Get the features in the test data but not in the training or validation data
uncommon_feature_test_not_train_val = np.setdiff1d(df_test.columns, df_common_var['common var'])

# Print the uncommon features
pd.DataFrame(uncommon_feature_test_not_train_val, columns=['uncommon feature'])

,uncommon feature


**Removing uncommon features**

In [23]:
# Remove the uncommon features from the training data
df_train = df_train.drop(columns=uncommon_feature_train_not_val_test)

# Print the first 5 rows of df_train
df_train.head()

,make_year,mileage_kmpl,engine_cc,fuel_type,owner_count,price_usd,brand,transmission,color,service_history,accidents_reported,insurance_valid
0,2009,18.27,2500,Petrol,4,4616.22,Chevrolet,Manual,Silver,Partial,0,Yes
1,1999,12.45,1200,Diesel,3,3213.13,BMW,Manual,White,Full,0,Yes
2,2011,19.30,2500,Petrol,5,5898.49,Tesla,Automatic,Gray,NaN,1,Yes
3,2015,17.67,4000,Diesel,5,7293.11,Toyota,Automatic,Gray,Full,0,Yes
4,2012,20.94,2000,Diesel,2,8589.11,Toyota,Automatic,Black,Full,1,Yes


In [24]:
# Remove the uncommon features from the validation data
df_val = df_val.drop(columns=uncommon_feature_val_not_train_test)

# Print the first 5 rows of df_val
df_val.head()

,make_year,mileage_kmpl,engine_cc,fuel_type,owner_count,price_usd,brand,transmission,color,service_history,accidents_reported,insurance_valid
0,2015,18.00,1800,Electric,5,10688.22,Kia,Manual,White,Full,2,Yes
1,2013,23.95,3000,Diesel,4,10564.03,BMW,Manual,Gray,Partial,0,Yes
2,2020,19.67,1800,Petrol,3,7231.20,Honda,Automatic,Blue,Full,1,Yes
3,2009,23.26,1500,Petrol,2,7649.42,Tesla,Manual,White,Partial,0,Yes
4,2007,12.54,3000,Petrol,5,6610.00,Volkswagen,Manual,Silver,Full,1,No


In [26]:
# Remove the uncommon features from the test data
df_test = df_test.drop(columns=uncommon_feature_test_not_train_val)

# Print the first 5 rows of df_test
df_test.head()

,make_year,mileage_kmpl,engine_cc,fuel_type,owner_count,price_usd,brand,transmission,color,service_history,accidents_reported,insurance_valid
0,2002,11.44,800,Petrol,3,1000.00,Toyota,Manual,White,Partial,2,No
1,2022,24.36,1500,Diesel,5,6658.41,Chevrolet,Manual,Gray,Partial,0,Yes
2,2006,22.51,4000,Petrol,3,8904.83,Ford,Manual,White,NaN,1,No
3,1995,26.14,800,Petrol,4,3403.00,BMW,Automatic,Black,Full,1,Yes
4,2022,23.20,1200,Diesel,1,8327.32,Chevrolet,Automatic,Silver,Partial,1,Yes


**Handling identifiers**

**Combining the training, validation and test data**

The code below shows how to combine the training, validation and test da

In [27]:
# Combine df_train, df_val and df_test
df = pd.concat([df_train, df_val, df_test], sort=False)

**Identifying identifiers**

The code below shows how to find identifiers from data.

In [28]:
def id_checker(df, dtype='float'):
    """
    The identifier checker

    Parameters
    ----------
    df : dataframe
    dtype : the data type identifiers cannot have, 'float' by default
            i.e., if a feature has this data type, it cannot be an identifier

    Returns
    ----------
    The dataframe of identifiers
    """

    # Get the dataframe of identifiers
    df_id = df[[var for var in df.columns
                # If the data type is not dtype
                if (df[var].dtype != dtype
                    # If the value is unique for each sample
                    and df[var].nunique(dropna=True) == df[var].notnull().sum())]]

    return df_id

In [30]:
# Call id_checker on df
# See the implementation in pmlm_utilities.ipynb
df_id = id_checker(df)

# Print the first 5 rows of df_id
df_id.head()

""
0
1
2
3
4


**Removing identifiers**

The code below shows how to remove identifiers from data.

In [31]:
import numpy as np

# Remove identifiers from df_train
df_train.drop(columns=np.intersect1d(df_id.columns, df_train.columns), inplace=True)

# Remove identifiers from df_val
df_val.drop(columns=np.intersect1d(df_id.columns, df_val.columns), inplace=True)

# Remove identifiers from df_test
df_test.drop(columns=np.intersect1d(df_id.columns, df_test.columns), inplace=True)

**Handling missing data**

**Combining the training, validation and test data**

The code below shows how to combine the training, validation and test data.

In [32]:
# Combine df_train, df_val and df_test
df = pd.concat([df_train, df_val, df_test], sort=False)

**Identifying missing values**

The code below shows how to find variables with NaN, their proportion of NaN and data type.

In [33]:
def nan_checker(df):
    """
    The NaN checker

    Parameters
    ----------
    df : the dataframe

    Returns
    ----------
    The dataframe of variables with NaN, their proportion of NaN and data type
    """

    # Get the dataframe of variables with NaN, their proportion of NaN and data type
    df_nan = pd.DataFrame([[var, df[var].isna().sum() / df.shape[0], df[var].dtype]
                           for var in df.columns if df[var].isna().sum() > 0],
                          columns=['var', 'proportion', 'dtype'])

    # Sort df_nan in accending order of the proportion of NaN
    df_nan = df_nan.sort_values(by='proportion', ascending=False).reset_index(drop=True)

    return df_nan

In [34]:
# Call nan_checker on df
# See the implementation in pmlm_utilities.ipynb
df_nan = nan_checker(df)

# Print df_nan
df_nan

,var,proportion,dtype
0,service_history,0.2038,object


In [35]:
# Print the unique data type of variables with NaN
pd.DataFrame(df_nan['dtype'].unique(), columns=['dtype'])

,dtype
0,object


In [36]:
df_miss = df_nan[df_nan['dtype'] == 'object'].reset_index(drop=True)
df_miss

,var,proportion,dtype
0,service_history,0.2038,object


**Separating the training, validation and test data**

The code below shows how to separate the training, validation and test data.


In [37]:
#Separating the training data
df_train = df.iloc[:df_train.shape[0], :]

# Separating the validation data
df_val = df.iloc[df_train.shape[0]:df_train.shape[0] + df_val.shape[0], :]

# Separating the test data
df_test = df.iloc[df_train.shape[0] + df_val.shape[0]:, :]

In [38]:
# Print the dimension of df_train
pd.DataFrame([[df_train.shape[0], df_train.shape[1]]], columns=['# rows', '# columns'])

,# rows,# columns
0,6000,12


In [39]:
# Print the dimension of df_val
pd.DataFrame([[df_val.shape[0], df_val.shape[1]]], columns=['# rows', '# columns'])

,# rows,# columns
0,2000,12


In [40]:
# Print the dimension of df_test
pd.DataFrame([[df_test.shape[0], df_test.shape[1]]], columns=['# rows', '# columns'])

,# rows,# columns
0,2000,12


**Handling Missing Values by Adding an "Unknown" Category**

In [41]:
from sklearn.impute import SimpleImputer

si = SimpleImputer(missing_values=np.nan, strategy='constant', fill_value='Unknown')

df_train[df_miss['var']] = si.fit_transform(df_train[df_miss['var']])
df_val[df_miss['var']] = si.transform(df_val[df_miss['var']])
df_test[df_miss['var']] = si.transform(df_test[df_miss['var']])

**Encoding the data**

Combining the training, validation and test data

The code below shows how to combine the training, validation and test data.

In [42]:
# Combine df_train, df_val and df_test
df = pd.concat([df_train, df_val, df_test], sort=False)

# Print the unique data type of variables in df
pd.DataFrame(df.dtypes.unique(), columns=['dtype'])

,dtype
0,int64
1,float64
2,object


**Identifying categorical variables**

The code below shows how to find categorical variables (whose data type is dtype) and their number of unique value.

In [43]:
def cat_var_checker(df, dtype='object'):
    """
    The categorical variable checker

    Parameters
    ----------
    df : the dataframe
    dtype : the data type categorical variables should have, 'object' by default
            i.e., if a variable has this data type, it should be a categorical variable

    Returns
    ----------
    The dataframe of categorical variables and their number of unique value
    """

    # Get the dataframe of categorical variables and their number of unique value
    df_cat = pd.DataFrame([[var, df[var].nunique(dropna=False)]
                           # If the data type is dtype
                           for var in df.columns if df[var].dtype == dtype],
                          columns=['var', 'nunique'])

    # Sort df_cat in accending order of the number of unique value
    df_cat = df_cat.sort_values(by='nunique', ascending=False).reset_index(drop=True)

    return df_cat

In [44]:
# Call cat_var_checker on df
# See the implementation in pmlm_utilities.ipynb
df_cat = cat_var_checker(df)

# Print the dataframe
df_cat

,var,nunique
0,brand,10
1,color,6
2,fuel_type,3
3,service_history,3
4,transmission,2
5,insurance_valid,2


**Encoding categorical features**

The code below shows how to encode categorical features in the combined data.

In [45]:
# One-hot-encode the categorical features in the combined data
df = pd.get_dummies(df, columns=np.setdiff1d(df_cat['var'], [target]))

# Print the first 5 rows of df
df.head()

,make_year,mileage_kmpl,engine_cc,owner_count,price_usd,accidents_reported,brand_BMW,brand_Chevrolet,brand_Ford,brand_Honda,...,fuel_type_Diesel,fuel_type_Electric,fuel_type_Petrol,insurance_valid_No,insurance_valid_Yes,service_history_Full,service_history_Partial,service_history_Unknown,transmission_Automatic,transmission_Manual
0,2009,18.27,2500,4,4616.22,0,False,True,False,False,...,False,False,True,False,True,False,True,False,False,True
1,1999,12.45,1200,3,3213.13,0,True,False,False,False,...,True,False,False,False,True,True,False,False,False,True
2,2011,19.30,2500,5,5898.49,1,False,False,False,False,...,False,False,True,False,True,False,False,True,True,False
3,2015,17.67,4000,5,7293.11,0,False,False,False,False,...,True,False,False,False,True,True,False,False,True,False
4,2012,20.94,2000,2,8589.11,1,False,False,False,False,...,True,False,False,False,True,True,False,False,True,False


**Separating the training, validation and test data**

The code below shows how to separate the training, validation and test data.

In [46]:
# Separating the training data
df_train = df.iloc[:df_train.shape[0], :]

# Separating the validation data
df_val = df.iloc[df_train.shape[0]:df_train.shape[0] + df_val.shape[0], :]

# Separating the test data
df_test = df.iloc[df_train.shape[0] + df_val.shape[0]:, :]

In [47]:
# Print the dimension of df_train
pd.DataFrame([[df_train.shape[0], df_train.shape[1]]], columns=['# rows', '# columns'])

,# rows,# columns
0,6000,32


In [48]:
# Print the dimension of df_val
pd.DataFrame([[df_val.shape[0], df_val.shape[1]]], columns=['# rows', '# columns'])

,# rows,# columns
0,2000,32


In [49]:
# Print the dimension of df_test
pd.DataFrame([[df_test.shape[0], df_test.shape[1]]], columns=['# rows', '# columns'])

,# rows,# columns
0,2000,32


**Splitting the feature and target**

The code below shows how to split the feature and target.

In [50]:
# Get the feature matrix
X_train = df_train[np.setdiff1d(df_train.columns, [target])].values
X_val = df_val[np.setdiff1d(df_val.columns, [target])].values
X_test = df_test[np.setdiff1d(df_test.columns, [target])].values

# Get the target vector
y_train = df_train[target].values
y_val = df_val[target].values
y_test = df_test[target].values

**Scaling the data**

**Standardization **

The code below shows how to standardize the data.

In [51]:
from sklearn.preprocessing import StandardScaler

# The StandardScaler
ss = StandardScaler()

**Standardizing the features**

The code below shows how to standardize the features.

In [52]:
# Standardize the training data
X_train = ss.fit_transform(X_train)

# Standardize the validation data
X_val = ss.transform(X_val)

# Standardize the test data
X_test = ss.transform(X_test)

**Standardizing the target**

The code below shows how to standardize the features.

In [53]:
# Standardize the training data
y_train = ss.fit_transform(y_train.reshape(-1, 1)).reshape(-1)

# Standardize the validation data
y_val = ss.transform(y_val.reshape(-1, 1)).reshape(-1)

# Standardize the test data
y_test = ss.transform(y_test.reshape(-1, 1)).reshape(-1)

**Getting the predefined split cross-validator**

In [55]:
from sklearn.model_selection import PredefinedSplit

def get_train_val_ps(X_train, y_train, X_val, y_val):
    """
    Get the:
    feature matrix and target velctor in the combined training and validation data
    target vector in the combined training and validation data
    PredefinedSplit

    Parameters
    ----------
    X_train : the feature matrix in the training data
    y_train : the target vector in the training data
    X_val : the feature matrix in the validation data
    y_val : the target vector in the validation data

    Return
    ----------
    The feature matrix in the combined training and validation data
    The target vector in the combined training and validation data
    PredefinedSplit
    """

    # Combine the feature matrix in the training and validation data
    X_train_val = np.vstack((X_train, X_val))

    # Combine the target vector in the training and validation data
    y_train_val = np.vstack((y_train.reshape(-1, 1), y_val.reshape(-1, 1))).reshape(-1)

    # Get the indices of training and validation data
    train_val_idxs = np.append(np.full(X_train.shape[0], -1), np.full(X_val.shape[0], 0))

    # The PredefinedSplit
    ps = PredefinedSplit(train_val_idxs)

    return X_train_val, y_train_val, ps

Hyperparameter Tuning

In [94]:
from sklearn.linear_model import SGDRegressor
from sklearn.linear_model import LinearRegression
from xgboost import XGBRegressor

models = {'sgd': SGDRegressor(random_state=random_seed),
          'lr': LinearRegression(),
          'xgb': XGBRegressor(random_state=random_seed)
}

**Creating the dictionary of the pipelines**

**In the dictionary:**

the key is the acronym of the model the value is the pipeline, which, for now, only includes the model

In [86]:
from sklearn.pipeline import Pipeline

pipes = {}

for acronym, model in models.items():
    pipes[acronym] = Pipeline([('model', model)])

**Getting the predefined split cross-validator**

In [87]:
# Get the:
# feature matrix and target velctor in the combined training and validation data
# target vector in the combined training and validation data
# PredefinedSplit
# See the implementation in pmlm_utilities.ipynb
X_train_val, y_train_val, ps = get_train_val_ps(X_train, y_train, X_val, y_val)

In [95]:
param_grids = {
    'lr': [{}],
    'xgb': [{
        'model__n_estimators': [100, 200, 300],
        'model__max_depth': [3, 5, 7],
        'model__learning_rate': [0.01, 0.05, 0.1]
    }],
    'sgd': [{
        'model__eta0': [0.0038, 0.0039, 0.0040, 0.0041, 0.0042],
        'model__alpha': [0.0002, 0.00025, 0.0003, 0.00035, 0.0004]
    }]
}


In [96]:
# Make directory
directory = os.path.dirname(abspath_curr + '/result/mnist/cv_results/GridSearchCV/')
if not os.path.exists(directory):
    os.makedirs(directory)

**Tuning the hyperparameters**

In [99]:
from sklearn.model_selection import GridSearchCV

# The list of [best_score_, best_params_, best_estimator_] obtained by GridSearchCV
best_score_params_estimator_gs = []

# For each model
for acronym in pipes.keys():
    # GridSearchCV
    gs = GridSearchCV(estimator=pipes[acronym],
                      param_grid=param_grids[acronym],
                      scoring='neg_mean_squared_error',
                      n_jobs=2,
                      cv=ps,
                      return_train_score=True)

    # Fit the pipeline
    gs = gs.fit(X_train_val, y_train_val)

    # Update best_score_params_estimator_gs
    best_score_params_estimator_gs.append([gs.best_score_, gs.best_params_, gs.best_estimator_])

    # Sort cv_results in ascending order of 'rank_test_score' and 'std_test_score'
    cv_results = pd.DataFrame.from_dict(gs.cv_results_).sort_values(by=['rank_test_score', 'std_test_score'])

    # Get the important columns in cv_results
    important_columns = ['rank_test_score',
                         'mean_test_score',
                         'std_test_score',
                         'mean_train_score',
                         'std_train_score',
                         'mean_fit_time',
                         'std_fit_time',
                         'mean_score_time',
                         'std_score_time']

    # Move the important columns ahead
    cv_results = cv_results[important_columns + sorted(list(set(cv_results.columns) - set(important_columns)))]

    # Write cv_results file
    cv_results.to_csv(path_or_buf=abspath_curr + '/result/mnist/cv_results/GridSearchCV/' + acronym + '.csv', index=False)

# Sort best_score_params_estimator_gs in descending order of the best_score_
best_score_params_estimator_gs = sorted(best_score_params_estimator_gs, key=lambda x : x[0], reverse=True)

# Print best_score_params_estimator_gs
pd.DataFrame(best_score_params_estimator_gs, columns=['best_score', 'best_param', 'best_estimator'])

,best_score,best_param,best_estimator
0,-0.129814,"{'model__alpha': 0.0002, 'model__eta0': 0.004}","(SGDRegressor(alpha=0.0002, eta0=0.004, random..."
1,-0.129924,{},(LinearRegression())
2,-0.134645,"{'model__learning_rate': 0.05, 'model__max_dep...","(XGBRegressor(base_score=None, booster=None, c..."
